In [1]:
from pytao import TaoModel
from pprint import pprint
import json
import os
import numpy as np
from math import sqrt

In [2]:
M = TaoModel('/Users/chrisonian/Code/GitHub/lcls-lattice/bmad/models/sc_hxr/tao.init')

Initialized Tao with /var/folders/wj/lfgr01993dx79p9cm_skykbw0000gn/T/tmp80r35bve/tao/tao.init


In [3]:
from pytao.tao_ctypes.util import parse_bool, parse_tao_lat_ele_list

In [4]:
SLIST = M.cmd_real('python lat_list 1@0>>*|model real:ele.s')
LLIST = M.cmd_real('python lat_list 1@0>>*|model real:ele.l')
NAMES = M.cmd('python lat_list  1@0>>*|model ele.name')

for s, l, n in zip(SLIST[0:20], LLIST[0:20], NAMES[0:20]):
    print (n, l, s)

BEGINNING 0.0 0.0
BEGGUNB 0.0 0.0
DGBCA -0.071705 -0.071705
SOL1BKB 0.0 -0.071705
DGBCB 0.071705 0.0
CATHODEB 0.0 0.0
DGUN 0.04 0.04
DG001 0.16348 0.20348
SOL1B 0.0861 0.28957999999999995
SOL1B#1 0.04304999999999999 0.24652999999999997
SQ01B 0.0 0.24652999999999997
CQ01B 0.0 0.24652999999999997
SOL1B#2 0.043050000000000005 0.28957999999999995
DG002 0.09789999999999999 0.38747999999999994
VV01B 0.0 0.38747999999999994
DG003 0.09702 0.48449999999999993
XC01B 0.0 0.48449999999999993
YC01B 0.0 0.48449999999999993
DG004 0.00515 0.4896499999999999
BPM1B 0.0 0.4896499999999999


In [5]:
# Index lookup function
ix_of = parse_tao_lat_ele_list(M.cmd('python lat_ele_list 1@0'))
s_of = {}
for n,s in zip(NAMES, SLIST):
    s_of[n] = s

In [7]:
CAVS = [name for name in NAMES if name.startswith('CAV') and ('#' not in name)]
KLYS = {}
for cav in CAVS:
    name, section = cav[0:-1], cav[-1:]
    if name not in KLYS:
        KLYS[name] = []
    KLYS[name].append(section)
len(CAVS)

296

# Linac overlays

L1 is CM 02 - 03

L2 is CM 04 - 15

L3 is CM 16 - 35

prefix: CAVL
Each CM has cavities like CAVL157 for CM 15, Cavity 7. 


3.9 GHz cavities
L1H is CMH1 - CMH2

prefix: CAVC

In [8]:
for i in range(10):
    print(f'{i:02d}') 

00
01
02
03
04
05
06
07
08
09


In [10]:
def CM_eles(ix, prefix='CAVL'):
    """Get eles for cyromodule"""
    return [f'{prefix}{ix:02d}{i}' for i in range(1,9)]
CM_eles(22)    

['CAVL221',
 'CAVL222',
 'CAVL223',
 'CAVL224',
 'CAVL225',
 'CAVL226',
 'CAVL227',
 'CAVL228']

In [11]:
# Define eles by linac
def Linac_eles(i1, i2, prefix='CAVL'):
    eles = []
    for i in range(i1, i2+1):
        eles += CM_eles(i, prefix=prefix)
    return eles

L0 = Linac_eles(1,1)
L1 = Linac_eles(2,3)
L2 = Linac_eles(4,15)
L3 = Linac_eles(16, 35)
L1H = Linac_eles(1,2, prefix='CAVC')


In [12]:
# Make overlays
# Voltage actually controls gradient
def phase_overlay(name, cavs):
    lines = []
    lines.append('!--------------\n')
    lines.append('! Linac phase and voltage overlay\n')
    lines.append(f'{name}: overlay = {{\n')
    i=0
    for n in cavs:
        i += 1
        lines.append(f'  {n}[phi0]:phase_deg/360, {n}[gradient]:voltage/{n}[L],')
        if i == 2:
            i = 0
            lines.append('\n')   
    if lines[-1] == '\n':
        lines.pop()
        
    lines[-1] = lines[-1][:-1]+'}, var = {phase_deg, voltage}\n\n'
    return ''.join(lines)
print(phase_overlay('SC_L1',L1)           )

!--------------
! Linac phase and voltage overlay
SC_L1: overlay = {
  CAVL021[phi0]:phase_deg/360, CAVL021[gradient]:voltage/CAVL021[L],  CAVL022[phi0]:phase_deg/360, CAVL022[gradient]:voltage/CAVL022[L],
  CAVL023[phi0]:phase_deg/360, CAVL023[gradient]:voltage/CAVL023[L],  CAVL024[phi0]:phase_deg/360, CAVL024[gradient]:voltage/CAVL024[L],
  CAVL025[phi0]:phase_deg/360, CAVL025[gradient]:voltage/CAVL025[L],  CAVL026[phi0]:phase_deg/360, CAVL026[gradient]:voltage/CAVL026[L],
  CAVL027[phi0]:phase_deg/360, CAVL027[gradient]:voltage/CAVL027[L],  CAVL028[phi0]:phase_deg/360, CAVL028[gradient]:voltage/CAVL028[L],
  CAVL031[phi0]:phase_deg/360, CAVL031[gradient]:voltage/CAVL031[L],  CAVL032[phi0]:phase_deg/360, CAVL032[gradient]:voltage/CAVL032[L],
  CAVL033[phi0]:phase_deg/360, CAVL033[gradient]:voltage/CAVL033[L],  CAVL034[phi0]:phase_deg/360, CAVL034[gradient]:voltage/CAVL034[L],
  CAVL035[phi0]:phase_deg/360, CAVL035[gradient]:voltage/CAVL035[L],  CAVL036[phi0]:phase_deg/360, CAVL036[gr

# Get defaults

In [13]:
from pytao.util.elements import lat_element
from pytao.util.parameters import tao_parameter_dict

In [14]:
dat = M.cmd('python ele:gen_attribs 1@0>>1491|model')
params = tao_parameter_dict(dat)
params

OrderedDict([('L', L;REAL;True;0.14944),
             ('units#L', units#L;STR;False;m),
             ('TILT', TILT;REAL;True;0.0),
             ('units#TILT', units#TILT;STR;False;rad),
             ('X_PITCH', X_PITCH;REAL;True;0.0),
             ('units#X_PITCH', units#X_PITCH;STR;False;),
             ('Y_PITCH', Y_PITCH;REAL;True;0.0),
             ('units#Y_PITCH', units#Y_PITCH;STR;False;),
             ('X_OFFSET', X_OFFSET;REAL;True;0.0),
             ('units#X_OFFSET', units#X_OFFSET;STR;False;m),
             ('Y_OFFSET', Y_OFFSET;REAL;True;0.0),
             ('units#Y_OFFSET', units#Y_OFFSET;STR;False;m),
             ('Z_OFFSET', Z_OFFSET;REAL;True;0.0),
             ('units#Z_OFFSET', units#Z_OFFSET;STR;False;m),
             ('DELTA_REF_TIME', DELTA_REF_TIME;REAL;False;4.9847818991328e-10),
             ('units#DELTA_REF_TIME', units#DELTA_REF_TIME;STR;False;sec),
             ('P0C', P0C;REAL;False;3279999960.0848),
             ('units#P0C', units#P0C;STR;False;eV),
   

In [15]:
# Get all parameters for cavities
CAVDAT = {}
for name in CAVS:
    ix = ix_of[name]
    #print(name, ix)
    dat = M.cmd(f'python ele:gen_attribs 1@0>>{ix}|design')
    params = tao_parameter_dict(dat)
    CAVDAT[name] = params
#CAVDAT['CAV318']['VOLTAGE'].value

In [16]:
def defaults(eles, param='VOLTAGE'):
    vals = list(set(CAVDAT[C.upper()][param.upper()].value for C in eles))
    assert len(vals) == 1
    return vals[0]
defaults(L1, 'VOLTAGE'), defaults(L1, 'phi0')      
defaults(L1H, 'gradient'), defaults(L1H, 'phi0')      

(9053035.0, -0.41380555555556)

In [17]:
set([CAVDAT[C]['VOLTAGE'].value for C in L1]), set([CAVDAT[C]['VOLTAGE'].value for C in L2]), set([CAVDAT[C]['VOLTAGE'].value for C in L3])

({11445054.227484}, {15260716.509783}, {14999999.999996})

In [18]:
DEFAULTS=f"""
! Design linac phasing and voltages

SC_L1[voltage] =   {defaults(L1, 'voltage')}
SC_L1[phase_deg] = {defaults(L1, 'phi0')*360}

SC_L1H[voltage] =   {defaults(L1H, 'voltage')}
SC_L1H[phase_deg] = {defaults(L1H, 'phi0')*360}

SC_L2[voltage] =   {defaults(L2, 'voltage')}
SC_L2[phase_deg] = {defaults(L2, 'phi0')*360}

SC_L3[voltage] =   {defaults(L3, 'voltage')}
SC_L3[phase_deg] = {defaults(L3, 'phi0')*360}

"""
print(DEFAULTS)


! Design linac phasing and voltages

SC_L1[voltage] =   11445054.227484
SC_L1[phase_deg] = -19.19999999999988

SC_L1H[voltage] =   3131574.9403962
SC_L1H[phase_deg] = -148.9700000000016

SC_L2[voltage] =   15260716.509783
SC_L2[phase_deg] = -20.750000000000043

SC_L3[voltage] =   14999999.999996
SC_L3[phase_deg] = 0.0




In [19]:
lines = []
for c in L1+L1H+L2+L3:
    lines.append(f'{c}[field_master]=T')
FIELD_MASTER = '\n'.join(lines)

# Write

In [20]:
with open('../linac_overlays.bmad', 'w') as f:
    #f.write(FIELD_MASTER+'\n')
    f.write(phase_overlay('SC_L1',L1))
    f.write(phase_overlay('SC_L1H',L1H))
    f.write(phase_overlay('SC_L2',L2))
    f.write(phase_overlay('SC_L3',L3))
    f.write(DEFAULTS)   
    
    

In [21]:
134.771785 - 132.324469

2.4473160000000007

In [25]:
Ltot = 2.447316
Lp = Ltot*np.cos(-0.100774150058)
Lp

2.4348997407105535

In [29]:
Ltot = 3.53812662499103681E+02 - 3.43942942855998297E+02
Lp = Ltot*np.cos(0.043924602172)
Lp

9.860200000000017